# The speckle layer in code

[Notebook 5](05_Speckles_from_First_Principles) built the physics of residual
starlight from first principles. This notebook drives the software that puts
that physics to work: the `linearize` -> `(E_nom, G)` reduction, the closed-form
`stats`, and the Monte-Carlo `SpeckleProcess` generator. The through-line is that
the closed form and the generator are two views of one linear model, and they
agree.

A path is linear in the field, so around an operating point the focal field is
an affine function of a small wavefront perturbation `eps`:

$$ E(\varepsilon) = E_\mathrm{nom} + G\,\varepsilon, \qquad
G_k = \frac{\partial E_\mathrm{focal}}{\partial \varepsilon_k}. $$

`E_nom` is the deterministic coronagraphic leakage; each column `G_k` is the
focal-field response to one wavefront mode. That single pair generates every
speckle quantity below.

In [ ]:
import jax

jax.config.update("jax_enable_x64", True)  # the coherent cross term needs it

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LogNorm

import physicaloptix as po
from physicaloptix import (
    Field,
    Fraunhofer,
    Grid,
    MultiScaleVortex,
    OpticalPath,
    PlaneKind,
    SampledOptic,
    Stage,
    fourier_dm_basis,
    linearize,
    stats,
)

print("physicaloptix", po.__version__)

## The path we linearize

We reuse the charge-2 vortex coronagraph from [notebook 3](03_Building_a_Coronagraph):
a vortex focal-plane mask, a Lyot stop, and a final transform to the science
focal plane. The bare telescope gives the normalization -- the peak intensity
that maps to unit contrast.

In [ ]:
NPUP, NFOC, PSCALE = 64, 128, 0.25  # pupil, focal npix, and lambda/D per pixel
WL = 500.0  # design wavelength [nm]

pupil = Grid.pupil(NPUP)
focal = Grid.focal(NFOC, PSCALE)
x = np.asarray(pupil.coords)
xg, yg = np.meshgrid(x, x)

aperture = ((xg**2 + yg**2) <= 0.25).astype(complex)  # clear disk, radius 0.5
lyot = ((xg**2 + yg**2) <= 0.42**2).astype(float)
flat = Field(data=jnp.asarray(aperture), grid=pupil, plane=PlaneKind.PUPIL)

corona = OpticalPath(
    stages=(
        Stage(
            "vortex",
            MultiScaleVortex.build(
                charge=2, npup=NPUP, q=64, scaling_factor=4, window_size=16
            ),
        ),
        Stage(
            "lyot",
            SampledOptic(
                transmission=jnp.asarray(lyot), grid=pupil, plane=PlaneKind.PUPIL
            ),
        ),
        Stage("sci", Fraunhofer(grid_in=pupil, grid_out=focal)),
    )
)

telescope = OpticalPath(
    stages=(Stage("sci", Fraunhofer(grid_in=pupil, grid_out=focal)),)
)
airy, _ = telescope.propagate(flat)
PEAK = float(jnp.max(jnp.abs(airy.data) ** 2))  # unit-contrast normalization
print(f"telescope PSF peak = {PEAK:.4g}")

## Linearizing: one call, the whole model

`linearize(path, field, basis, wavelength_nm=...)` propagates the operating
point once to get `E_nom`, then builds the sensitivity stack `G` one column per
basis mode. For an OPD basis the column is exact and analytic,
$G_k = \mathrm{i}\,(2\pi/\lambda)\,\mathrm{Path}(B_k\,E_\mathrm{in})$, so no
finite differencing is involved.

We perturb with a band-limited Fourier (deformable-mirror) basis over 2 to 9
$\lambda/D$: exactly the spatial frequencies that scatter starlight into that
annulus of the focal plane.

In [ ]:
basis = fourier_dm_basis(pupil, n_actuators=20, k_min=2.0, k_max=9.0, rms_nm=1.0)
lin = linearize(corona, flat, basis, wavelength_nm=WL)

print(f"basis kind      : {basis.kind}")
print(f"modes (columns) : {lin.n_modes}")
print(f"E_nom           : {lin.e_nom.shape} {lin.e_nom.dtype}")
print(f"G               : {lin.G.shape} {lin.G.dtype}")
print(f"pixel scale     : {lin.pixel_scale_lod} lambda/D per pixel")

`E_nom` is complex -- it carries the leakage field's phase, not just its
intensity -- which is what makes the coherent pinning term below available at
all. Each column of `G` is a full focal field. Their squared magnitudes are the
per-mode speckle PSFs: a mode at frequency $k$ cycles per aperture lights up a
speckle at radius $k\,\lambda/D$.

In [ ]:
ext = [-NFOC / 2 * PSCALE, NFOC / 2 * PSCALE] * 2
enom_c = np.asarray(jnp.abs(lin.e_nom) ** 2) / PEAK

# three modes at increasing control frequency, by their focal-plane radius
r_mode = []
for k in range(lin.n_modes):
    gk = np.abs(np.asarray(lin.G[k])) ** 2
    yy, xx = np.unravel_index(int(gk.argmax()), gk.shape)
    r_mode.append(np.hypot((xx - NFOC / 2) * PSCALE, (yy - NFOC / 2) * PSCALE))
order = np.argsort(r_mode)
picks = [order[len(order) // 6], order[len(order) // 2], order[-len(order) // 6]]

fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
im0 = axes[0].imshow(
    enom_c,
    origin="lower",
    cmap="inferno",
    norm=LogNorm(1e-9, 1e-3),
    extent=ext,
    interpolation="none",
)
axes[0].set_title(r"$|E_\mathrm{nom}|^2$ (leakage)")
fig.colorbar(im0, ax=axes[0], fraction=0.046)
for ax, k in zip(axes[1:], picks):
    gk = np.abs(np.asarray(lin.G[k])) ** 2
    gk = gk / gk.max()
    ax.imshow(
        gk,
        origin="lower",
        cmap="viridis",
        norm=LogNorm(1e-4, 1),
        extent=ext,
        interpolation="none",
    )
    ax.set_title(rf"$|G_k|^2$ at {r_mode[k]:.1f} $\lambda/D$")
for ax in axes:
    ax.set_xlabel(r"$\lambda/D$")
plt.tight_layout()
plt.show()

## The closed form: coherent floor plus incoherent halo

With zero-mean, independent mode coefficients of per-mode variance
$\sigma_k^2$, the mean focal intensity splits cleanly into a deterministic
coherent part and a stochastic incoherent halo:

$$ I_c = |E_\mathrm{nom}|^2, \qquad
I_s = \sum_k \sigma_k^2\,|G_k|^2, \qquad
\langle I \rangle = I_c + I_s. $$

`stats.coherent_intensity` and `stats.incoherent_intensity` are those two sums,
as pure functions of the same `(E_nom, G)`. Here we spend a 5 nm total wavefront
budget spread over the modes.

In [ ]:
TOTAL_RMS = 5.0  # nm of wavefront error, in quadrature over modes
per_mode_rms = jnp.full((lin.n_modes,), TOTAL_RMS / np.sqrt(lin.n_modes))

Ic = np.asarray(stats.coherent_intensity(lin.e_nom)) / PEAK
Is = np.asarray(stats.incoherent_intensity(lin.G, per_mode_rms)) / PEAK
mean_I = Ic + Is

mask = np.asarray(stats.dark_zone_mask(focal, iwa_lod=3.0, owa_lod=9.0))
print(f"median contrast in the dark zone: {np.median(mean_I[mask]):.2e}")

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, img, title in zip(
    axes,
    (Ic, Is, mean_I),
    (
        r"$I_c=|E_\mathrm{nom}|^2$",
        r"$I_s=\sum_k\sigma_k^2|G_k|^2$",
        r"$\langle I\rangle=I_c+I_s$",
    ),
):
    im = ax.imshow(
        img,
        origin="lower",
        cmap="inferno",
        norm=LogNorm(1e-9, 1e-3),
        extent=ext,
        interpolation="none",
    )
    ax.set_title(title)
    ax.set_xlabel(r"$\lambda/D$")
    theta = np.linspace(0, 2 * np.pi, 200)
    for r in (3.0, 9.0):
        ax.plot(r * np.cos(theta), r * np.sin(theta), "c--", lw=0.7)
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

The coherent floor $I_c$ is the static leakage the design cannot remove; the
halo $I_s$ is the speckle field the wavefront error scatters into the control
annulus. Where they overlap, the two interfere -- which the mean does not show,
but the pixel statistics do.

## Pixel statistics: the modified Rician

At a single pixel the instantaneous intensity is not fixed at $I_c + I_s$; it
fluctuates as the speckle field drifts. Because the halo is coherent with the
leakage, the intensity follows the modified Rician distribution

$$ p(I) = \frac{1}{I_s}\,
\exp\!\left(-\frac{I + I_c}{I_s}\right)
I_0\!\left(\frac{2\sqrt{I\,I_c}}{I_s}\right), $$

whose bright tail is heavier than a Gaussian -- the statistical signature of
{term}`speckle pinning`. `stats.modified_rician_pdf` evaluates it. We compare it
against a histogram of instantaneous intensities from many independent draws of
the generator (below), at two pixels: one deep in the halo where $I_c \ll I_s$
(nearly exponential) and one near the bright leakage where $I_c \sim I_s$
(a pinned, heavy tail).

In [ ]:
# generator with a complex E_nom so draws carry the coherent cross term
proc = lin.to_speckle_process(
    normalization=PEAK, decorr_hours=2.0, total_rms=TOTAL_RMS, coherent=True
)

# a deep halo pixel (Ic << Is) and a pinned pixel (Ic and Is both bright)
deep = np.unravel_index(int(np.where(mask, Is, 0.0).argmax()), Is.shape)
pinned = np.unravel_index(int(np.where(mask, Ic * Is, 0.0).argmax()), Is.shape)

keys = jax.random.split(jax.random.PRNGKey(0), 4000)


def contrast_at(key, idx):
    field = proc.draw(key)
    return Ic[idx] + field.realize(wavelength_nm=WL, time_s=0.0)[idx]


fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, idx, label in ((axes[0], deep, "deep halo"), (axes[1], pinned, "pinned")):
    samples = np.asarray(jax.vmap(lambda k, idx=idx: contrast_at(k, idx))(keys))
    ic_p, is_p = float(Ic[idx]), float(Is[idx])
    grid_c = np.linspace(max(1e-12, samples.min()), samples.max(), 300)
    pdf = np.asarray(stats.modified_rician_pdf(jnp.asarray(grid_c), ic_p, is_p))
    ax.hist(samples, bins=40, density=True, color="0.75", label="draws")
    ax.plot(grid_c, pdf, "r-", lw=2, label="modified Rician")
    ax.axvline(ic_p + is_p, color="k", ls=":", lw=1, label=r"$I_c+I_s$")
    ax.set_title(rf"{label}: $I_c/I_s = {ic_p / is_p:.2f}$")
    ax.set_xlabel("contrast")
    ax.set_ylabel("density")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

The closed-form density and the sampled histogram lie on top of each other, at
both pixels, with nothing fit between them: the `stats` functions and the
`SpeckleProcess` generator are the same model. Where $I_c \sim I_s$ the tail
reaches well past the mean $I_c + I_s$ -- a pinned speckle spends part of its
time far brighter than its average, which is what makes pinning a detection
hazard rather than a bookkeeping detail.

## From parameters to realizations

`lin.to_speckle_process(...)` packaged `(E_nom, G)` with a temporal model. Its
`draw(key)` returns an `AnalyticSpeckleField`: a frozen realization whose mode
coefficients drift in time by spectral synthesis,
$\varepsilon_k(t) = \sum_j a_{kj}\cos(2\pi f_j t + \phi_{kj})$. Calling
`realize(wavelength_nm=..., time_s=...)` evaluates the speckle field at a time.
It returns the contrast *delta* -- the excess over the deterministic floor
$|E_\mathrm{nom}|^2$ -- so it adds cleanly on top of a coronagraph's static
leakage downstream. We watch one realization boil over two hours.

In [ ]:
field = proc.draw(jax.random.PRNGKey(7))
times_h = [0.0, 0.7, 1.4, 2.1]

fig, axes = plt.subplots(1, len(times_h), figsize=(14, 3.6))
for ax, t_h in zip(axes, times_h):
    delta = np.asarray(field.realize(wavelength_nm=WL, time_s=t_h * 3600.0))
    total = Ic + delta  # instantaneous contrast on top of the floor
    im = ax.imshow(
        total,
        origin="lower",
        cmap="inferno",
        norm=LogNorm(1e-9, 1e-3),
        extent=ext,
        interpolation="none",
    )
    ax.set_title(f"t = {t_h:.1f} h")
    ax.set_xlabel(r"$\lambda/D$")
fig.colorbar(im, ax=axes, fraction=0.02, pad=0.02, label="contrast")
plt.show()

The speckle pattern evolves but does not fully reshuffle between frames: the
knee frequency set by `decorr_hours` fixes how fast it decorrelates. That
temporal correlation is exactly what differential imaging leans on, and what a
wavefront-control loop has to keep up with.

## Coherent versus incoherent: pinning in the map

By default a drawn field returns the strictly positive incoherent halo
$|G\varepsilon|^2/\mathrm{norm}$. With `coherent=True` it adds the cross term
$2\,\mathrm{Re}(E_\mathrm{nom}^{*}\,G\varepsilon)$, the interference between the
leakage and the speckle field. That term is what pins speckles to the bright
leakage -- and it can go negative, digging some speckles below the incoherent
floor while brightening others. Same random draw, both switches:

In [ ]:
key = jax.random.PRNGKey(11)
proc_inc = lin.to_speckle_process(
    normalization=PEAK, decorr_hours=2.0, total_rms=TOTAL_RMS, coherent=False
)
proc_coh = lin.to_speckle_process(
    normalization=PEAK, decorr_hours=2.0, total_rms=TOTAL_RMS, coherent=True
)
inc = np.asarray(proc_inc.draw(key).realize(wavelength_nm=WL, time_s=0.0))
coh = np.asarray(proc_coh.draw(key).realize(wavelength_nm=WL, time_s=0.0))
cross = coh - inc  # the pinning term, 2 Re(E_nom* G eps)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
im0 = axes[0].imshow(
    inc,
    origin="lower",
    cmap="inferno",
    norm=LogNorm(1e-10, 1e-4),
    extent=ext,
    interpolation="none",
)
axes[0].set_title(r"incoherent $|G\varepsilon|^2$")
fig.colorbar(im0, ax=axes[0], fraction=0.046)
vmax = np.abs(cross).max()
im1 = axes[1].imshow(
    cross,
    origin="lower",
    cmap="RdBu_r",
    vmin=-vmax,
    vmax=vmax,
    extent=ext,
    interpolation="none",
)
axes[1].set_title(r"cross term $2\,\mathrm{Re}(E_\mathrm{nom}^* G\varepsilon)$")
fig.colorbar(im1, ax=axes[1], fraction=0.046)
im2 = axes[2].imshow(
    np.abs(coh),
    origin="lower",
    cmap="inferno",
    norm=LogNorm(1e-10, 1e-4),
    extent=ext,
    interpolation="none",
)
axes[2].set_title("coherent total (abs)")
fig.colorbar(im2, ax=axes[2], fraction=0.046)
for ax in axes:
    ax.set_xlabel(r"$\lambda/D$")
plt.tight_layout()
plt.show()

The cross term is largest where the leakage is bright (small radii): that is
the pinning. An intensity-only PSF table cannot produce it, because it has
thrown away the phase of $E_\mathrm{nom}$. Carrying the complex field is the
reason this generator exists.

## The generator and the closed form agree

The two views must be consistent: averaging many incoherent draws should
reproduce the closed-form halo $I_s$. This is the numerical check that the
Monte-Carlo path and the analytic path are one model, not two approximations
that happen to look alike.

In [ ]:
ens_keys = jax.random.split(jax.random.PRNGKey(21), 400)
ensemble = jax.vmap(lambda k: proc_inc.draw(k).realize(wavelength_nm=WL, time_s=0.0))(
    ens_keys
)
mc_halo = np.asarray(jnp.mean(ensemble, axis=0))

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(Is[mask], mc_halo[mask], s=4, alpha=0.3, color="C0")
lim = [Is[mask].min(), Is[mask].max()]
ax.plot(lim, lim, "k--", lw=1, label="y = x")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"closed form $I_s$ / peak")
ax.set_ylabel("Monte-Carlo mean / peak")
ax.set_title("400 draws vs the analytic halo")
ax.legend()
plt.tight_layout()
plt.show()

rel = np.abs(mc_halo[mask] - Is[mask]).mean() / Is[mask].mean()
print(f"mean relative error over the dark zone: {rel:.1%}")

## Where this plugs in

- `AnalyticSpeckleField` implements optixstuff's `AbstractSpeckleField`, so
  [coronagraphoto](https://coronagraphoto.readthedocs.io/) consumes a drawn
  field directly as the time-varying speckle term of an image.
- The same `(E_nom, G)` product is what the
  [tiptilt](https://github.com/CoreySpohn/tiptilt) library builds on: it drives
  the linear model to generate drifting aberrations and to close a dark-hole
  control loop.
- The [architecture page](../explanation/architecture) places the speckle layer
  in the wider stack, and [notebook 5](05_Speckles_from_First_Principles) is the
  physics behind every quantity used here.